In [ ]:
import importlib

import mlflow.pyfunc
import pandas as pd
import scipy.stats

from sportsml.graph.graph import create_complete_graph


In [ ]:
SPORT = "cbb"

features_module = importlib.import_module(f"sportsml.{SPORT}.data.features")
team_name_map = importlib.import_module(f"sportsml.{SPORT}.data.nodes").team_name_map

In [ ]:
predictor = mlflow.pyfunc.load_model(f'../models/{SPORT}/pyg')

In [ ]:
gp = create_complete_graph(predictor.unwrap_python_model().team_embeddings.shape[0])
model_input = pd.DataFrame({
    'team': gp.edge_index[1],
    'opp': gp.edge_index[0],
})

In [ ]:
preds = predictor.predict(model_input)
preds['win_prob'] = scipy.stats.norm.cdf(preds['preds'] / preds['preds'].std())

In [ ]:
preds = preds.pivot_table(values='preds', index='team', columns='opp')

In [ ]:
preds = preds.rename(index=team_name_map, columns=team_name_map)

In [ ]:
preds = preds.loc[preds.mean(axis=0).sort_values().index, preds.mean(axis=0).sort_values().index]

In [ ]:
preds